# Advanced 4 — Conformal prediction (гарантована покривність)

**Split-conformal** дає множини-прогнози з **маржинальною гарантією покриття** ≥ 1−α без припущень про модель. Множина `{0,1}` = «обидва правдоподібні» → природна **абстенція** ('Unsure'), `{}` = «жоден» → теж Unsure. Реалізуємо вручну (без важких залежностей).

> Залежності: `%run 03_data_prep.ipynb`.

In [ ]:
%run 03_data_prep.ipynb

### Калібрування на окремому зрізі train + множини-прогнози на test

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

Xtr, ytr, Xte, yte = b['X_train'], b['y_train'], b['X_test'], b['y_test']
# proper-train (навчання) + calibration (для conformal-порогу)
Xpt, Xcal, ypt, ycal = train_test_split(Xtr, ytr, test_size=0.3, stratify=ytr, random_state=42)
clf = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                ('clf',LogisticRegression(max_iter=2000,C=0.5,random_state=42))]).fit(Xpt, ypt)

alpha = 0.10                                  # ціль: покриття ≥ 90%
Pcal = clf.predict_proba(Xcal)
# нонконформність істинної мітки = 1 − p(true)
scores = 1 - Pcal[np.arange(len(ycal)), ycal.to_numpy()]
n = len(scores)
qhat = np.quantile(scores, min(1.0, np.ceil((n+1)*(1-alpha))/n), method='higher')
print(f'conformal поріг q̂ = {qhat:.3f} (alpha={alpha})')

Pte = clf.predict_proba(Xte)
sets = [{k for k in (0,1) if (1 - Pte[i,k]) <= qhat} for i in range(len(Xte))]
cover = np.mean([yte.iloc[i] in sets[i] for i in range(len(Xte))])
sizes = np.array([len(s) for s in sets])
print(f'емпіричне покриття на test: {cover:.3f}  (ціль ≥ {1-alpha:.2f})')
print(f'розмір множин: |{{}}|={np.mean(sizes==0):.2f}  |1 клас|={np.mean(sizes==1):.2f}  |2 класи|={np.mean(sizes==2):.2f}')

### Зіставлення множин з рішенням Post / Do not post / Unsure

In [ ]:
def to_decision(s):
    if s == {1}: return 'Post'
    if s == {0}: return 'Do not post'
    return 'Unsure'        # {0,1} (обидва) або {} (жоден)
import collections
dec = collections.Counter(to_decision(s) for s in sets)
print('розподіл рішень (conformal):', dict(dec))

### Висновок (чесна інтерпретація)

Емпіричне покриття на test тут **втрималось** (~0.93 ≥ 0.90), хоча conformal гарантує його лише за **обмінюваності** (exchangeability), яку темпоральний дрейф порушує — тож на іншому зрізі воно могло б просісти; покладатись на гарантію під дрейфом не можна.

Ключове: покриття досягається **ціною абстенції** — ~**62%** прогнозів є повна множина `{0,1}` → 'Unsure', і лише ~2% — впевнений 'Post'. Тобто чесний conformal на слабкому сигналі переважно каже «не знаю» — що повністю узгоджено з духом проєкту (краще абстейнити, ніж вдавати впевненість). Для надійних гарантій під дрейфом — **weighted / time-series conformal** (наступний крок).